# Uncertainty-Calibrated Hierarchical Gaussian Processes for Intrusion Detection with Multi-Scale Temporal Modeling

## Reference Implementation — v3 (Revised Manuscript)

This notebook is the **complete, reproducible implementation** accompanying the Neurocomputing submission. It implements every component described in the paper:

| Component | Paper Reference | Module |
|-----------|----------------|--------|
| Deep-Kernel Projection π(·) | Section IV.A, Eq. 25-27 | `hgp_idps.models.projection_network` |
| Hierarchical GP Decomposition | Proposition 1 | `hgp_idps.models.hierarchical_gp` |
| Multi-Scale Temporal Kernels | Section IV.B, Eq. 33-35 | `hgp_idps.kernels.multiscale_temporal` |
| Domain-Specific Kernels | Appendix D.1-D.3 | `hgp_idps.kernels.domain_specific` |
| Kernel Centering (ANOVA) | Appendix D.4 | `hgp_idps.kernels.centered_kernel` |
| Adversarial Inducing Points | Algorithm 1, Appendix E | `hgp_idps.adversarial` |
| Uncertainty-Calibrated Detection | Section IV.D, Eq. 39-40 | `hgp_idps.detection.uncertainty` |
| 3-Tier Alert Prioritisation | Section IV.D.3 | `hgp_idps.detection.alert_prioritization` |
| Training Pipeline | Algorithm 2, Appendix H.3 | `hgp_idps.training.trainer` |
| Incremental Posterior Updates | Appendix G.1 | `hgp_idps.training.online_update` |
| Ablation & Transfer | Section VII.D-E | `hgp_idps.evaluation.ablation` |

**Dataset:** [ICS3D](https://doi.org/10.34740/kaggle/dsv/12483891) — 21.48 M records across Edge-IIoT, Container, and SOC domains.

---
## 0 — Environment Setup

In [ ]:
import sys, os
# Allow imports from the project root
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gpytorch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc

# ---------- reproducibility ----------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## 1 — Load & Preprocess ICS3D Data  (Section VI)

In [ ]:
from hgp_idps.data.loader import ICS3DDatasetLoader
from hgp_idps.data.preprocessing import preprocess_domain, create_unified_dataset

# Download and load all sub-datasets
loader = ICS3DDatasetLoader()
raw = loader.load(['edge_dnn', 'containers', 'soc_train', 'soc_test'])

In [ ]:
# Preprocess each domain
preprocessed = {}
scalers = {}

for name, df in raw.items():
    cfg = loader.DATASET_CONFIGS[name]
    X, y, feats, sc = preprocess_domain(
        df, label_col=cfg['label_col'], domain=cfg['domain']
    )
    preprocessed[name] = (X, y, feats)
    scalers[name] = sc
    print(f'  {name:15s}  X={X.shape}  attack_rate={y.mean():.2%}')

# Unified cross-domain dataset
X_all, y_all, domain_ids = create_unified_dataset(preprocessed)

---
## 2 — Train / Val / Test Split

In [ ]:
# Time-aware split (60 / 20 / 20) — preserves temporal ordering
n = len(X_all)
t1, t2 = int(0.6 * n), int(0.8 * n)

X_train, y_train = X_all[:t1], y_all[:t1]
X_val,   y_val   = X_all[t1:t2], y_all[t1:t2]
X_test,  y_test  = X_all[t2:], y_all[t2:]

print(f'Train : {X_train.shape[0]:>10,}  attack={y_train.mean():.2%}')
print(f'Val   : {X_val.shape[0]:>10,}  attack={y_val.mean():.2%}')
print(f'Test  : {X_test.shape[0]:>10,}  attack={y_test.mean():.2%}')

# Convert to tensors
Xt = torch.FloatTensor(X_train)
yt = torch.FloatTensor(y_train)
Xv = torch.FloatTensor(X_val)
yv = torch.FloatTensor(y_val)
Xe = torch.FloatTensor(X_test)
ye = torch.FloatTensor(y_test)

BATCH = 2048
train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=BATCH)
test_loader  = DataLoader(TensorDataset(Xe, ye), batch_size=BATCH)

---
## 3 — Build Hierarchical Kernel  (Section IV.B)

Additive decomposition with centered components:

$$k = \tilde{k}_{\text{shared}} + \tilde{k}_{\text{domain}} + \sum_j \tilde{k}_{\text{temporal},j} + \tilde{k}_{\text{interaction}}$$

In [ ]:
from gpytorch.kernels import ScaleKernel, RBFKernel, MaternKernel
from gpytorch.priors import LogNormalPrior

from hgp_idps.kernels.multiscale_temporal import MultiScaleTemporalKernel
from hgp_idps.kernels.centered_kernel import CenteredAdditiveKernel

# Latent dim from projection network
LATENT_DIM = 32

# --- Shared kernel (RBF-ARD) ---
shared = ScaleKernel(
    RBFKernel(ard_num_dims=LATENT_DIM,
              lengthscale_prior=LogNormalPrior(0.0, 1.0))
)

# --- Domain kernel (Matern-2.5) ---
domain_k = ScaleKernel(
    MaternKernel(nu=2.5, lengthscale_prior=LogNormalPrior(0.0, 1.0))
)

# --- Multi-scale temporal (7 RBF + 3 periodic + spectral + changepoint) ---
temporal = MultiScaleTemporalKernel(
    use_spectral=True, num_mixtures=4, use_changepoint=True
)

# --- Interaction kernel ---
interaction = ScaleKernel(
    RBFKernel(lengthscale_prior=LogNormalPrior(0.5, 0.3))
)

# Collect all components for centering
kernel_components = [
    ('shared_rbf', shared),
    ('domain_matern', domain_k),
] + temporal.get_sub_kernels() + [
    ('interaction', interaction),
]

# Build centered additive kernel
full_kernel = CenteredAdditiveKernel(kernel_components)
print(f'Kernel components: {len(kernel_components)}')
for name, _ in kernel_components:
    print(f'  - {name}')

---
## 4 — Build & Initialise Model  (Section IV.A + Algorithm 1)

In [ ]:
from hgp_idps.training.trainer import HierarchicalGPTrainer

# Training configuration matching paper Appendix H.3
train_cfg = {
    'epochs': 100,
    'gradient_clip': 1.0,
    'gradient_accumulation_steps': 4,
    'early_stopping': {'patience': 10},
    'validation_every': 5,
    'adversarial_frequency': 5,
    'optimizer': {
        'variational': {'lr': 1e-3, 'betas': [0.9, 0.999]},
    },
    'scheduler': {'T_0': 20, 'T_mult': 2, 'eta_min': 1e-5},
}

trainer = HierarchicalGPTrainer(config=train_cfg, device=DEVICE)

# Build model with adversarial inducing-point init
model = trainer.build_model(
    input_dim=X_all.shape[1],
    kernel=full_kernel,
    X_train=Xt,
    y_train=yt,
    num_inducing=500,
    projection_cfg={'layers': [128, 64, 32], 'activation': 'silu',
                    'dropout': 0.1, 'batch_norm': True},
    adversarial_cfg={'enabled': True, 'epsilon': 0.01, 'pgd_steps': 10},
)

---
## 5 — Train  (Algorithm 2)

In [ ]:
trainer.train(train_loader, val_loader)

---
## 6 — Evaluate on Test Set  (Section VII)

In [ ]:
from hgp_idps.detection.uncertainty import UncertaintyCalibratedDetector
from hgp_idps.detection.alert_prioritization import AlertPrioritizer
from hgp_idps.evaluation.metrics import compute_all_metrics, expected_calibration_error

# Detector
detector = UncertaintyCalibratedDetector(model, config={
    'uncertainty_weight': 0.5,
    'entropy_weight': 0.3,
    'adaptive_threshold': True,
    'base_threshold': 0.5,
})

# Calibrate baseline on training normal traffic
normal_idx = yt == 0
normal_sample = Xt[normal_idx][:2000].to(DEVICE)
detector.update_baseline(normal_sample)

# Run detection on test set
all_preds, all_scores, all_conf = [], [], []

model.eval()
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(DEVICE)
        res = detector.detect(bx, domain='multi')
        all_preds.append(res['detections'].cpu())
        all_scores.append(res['scores'].cpu())
        all_conf.append(res['confidence'].cpu())

preds_np  = torch.cat(all_preds).numpy()
scores_np = torch.cat(all_scores).numpy()
conf_np   = torch.cat(all_conf).numpy()
y_test_np = ye.numpy()

# Core metrics
metrics = compute_all_metrics(y_test_np, preds_np, scores_np)
print('\n  DETECTION METRICS')
print('  ' + '-' * 30)
for k, v in metrics.items():
    print(f'    {k:12s}: {v:.4f}')

# Calibration
cal = expected_calibration_error(y_test_np, conf_np, preds_np, n_bins=15)
print(f'\n    ECE         : {cal["ece"]:.4f}')

---
## 7 — Adversarial Robustness Evaluation  (Table VII)

In [ ]:
from hgp_idps.adversarial.pgd_attack import pgd_attack
from sklearn.metrics import accuracy_score

# Subsample test set for speed
n_adv = min(5000, Xe.size(0))
X_adv_base = Xe[:n_adv].to(DEVICE)
y_adv_base = ye[:n_adv].to(DEVICE)

epsilons = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]
adv_results = {}

for eps in epsilons:
    if eps == 0.0:
        X_in = X_adv_base
    else:
        X_in = pgd_attack(model, X_adv_base, y_adv_base, epsilon=eps, steps=10)
    
    res = detector.detect(X_in, domain='multi')
    acc = accuracy_score(y_adv_base.cpu().numpy(),
                         res['detections'].cpu().numpy())
    adv_results[eps] = acc
    print(f'  eps={eps:.3f}  accuracy={acc:.4f}')

---
## 8 — Kernel Variance Attribution  (Appendix D.4)

In [ ]:
# Project test data into latent space for attribution
with torch.no_grad():
    Z_test = model.projection(Xe[:2000].to(DEVICE))

kernel_attr = model.gp.covar_module.variance_attribution(Z_test)
print('\n  KERNEL VARIANCE ATTRIBUTION')
print('  ' + '-' * 40)
for name, frac in sorted(kernel_attr.items(), key=lambda x: -x[1]):
    print(f'    {name:30s}: {frac:.3f}')

---
## 9 — 3-Tier Alert Prioritisation  (Section IV.D.3)

In [ ]:
# Run on a sample batch
sample_X = Xe[:1000].to(DEVICE)
sample_res = detector.detect(sample_X, domain='multi')

prioritizer = AlertPrioritizer(sigma_thresholds=(1.0, 2.0))
alerts = prioritizer.prioritize(sample_res)
alert_summary = prioritizer.summary(alerts)

print('\n  ALERT TIER DISTRIBUTION (sample of 1000)')
print('  ' + '-' * 30)
for tier, count in alert_summary.items():
    print(f'    {tier:10s}: {count}')
print(f'    {"TOTAL":10s}: {sum(alert_summary.values())}')

---
## 10 — Ablation Study  (Section VII.D)

In [ ]:
from hgp_idps.evaluation.ablation import run_ablation_study

ablation_components = [
    'shared_rbf', 'domain_matern',
    'rbf_microsecond', 'rbf_second', 'rbf_hour', 'rbf_week',
    'periodic_daily', 'spectral_mixture', 'changepoint',
    'interaction',
]

ablation = run_ablation_study(
    trainer, test_loader, ablation_components, domain='multi'
)

---
## 11 — Cross-Domain Transfer  (Section VII.E)

In [ ]:
from hgp_idps.evaluation.ablation import run_cross_domain_transfer

# Transfer: multi-domain → SOC (most challenging, 99:1 imbalance)
if 'soc_test' in preprocessed:
    X_soc, y_soc, _ = preprocessed['soc_test']
    # Pad to unified dimension
    if X_soc.shape[1] < X_all.shape[1]:
        pad = np.zeros((X_soc.shape[0], X_all.shape[1] - X_soc.shape[1]), dtype=np.float32)
        X_soc = np.hstack([X_soc, pad])
    
    transfer = run_cross_domain_transfer(
        trainer,
        torch.FloatTensor(X_soc),
        torch.FloatTensor(y_soc),
        target_domain='soc',
        fine_tune_epochs=20,
        fine_tune_fraction=0.1,
    )

---
## 12 — Incremental / Online Posterior Update  (Appendix G.1)

In [ ]:
from hgp_idps.training.online_update import IncrementalPosteriorUpdater

updater = IncrementalPosteriorUpdater(model, base_lr=0.01)

# Simulate streaming: 5 mini-batches of 64 new observations
for i in range(5):
    idx = np.random.choice(len(Xe), 64, replace=False)
    x_new = Xe[idx].to(DEVICE)
    y_new = ye[idx].to(DEVICE)
    updater.update(x_new, y_new, domain='multi')
    print(f'  Online update batch {i+1}/5 complete')

# Re-evaluate after online updates
model.eval()
post_update_preds = []
with torch.no_grad():
    for bx, _ in test_loader:
        bx = bx.to(DEVICE)
        out = model(bx)
        post_update_preds.append(model.likelihood(out).mean.round().cpu())

post_preds = torch.cat(post_update_preds).numpy()
post_metrics = compute_all_metrics(y_test_np, post_preds, scores_np)
print(f'\n  Post-update F1: {post_metrics["f1"]:.4f}')

---
## 13 — Comprehensive Visualisations  (Section VII Figures)

In [ ]:
from hgp_idps.evaluation.visualization import ResultVisualizer

# Prepare ROC data
if len(np.unique(y_test_np)) > 1:
    fpr_arr, tpr_arr, _ = roc_curve(y_test_np, scores_np)
    roc_data = {'fpr': fpr_arr, 'tpr': tpr_arr, 'auc': auc(fpr_arr, tpr_arr)}
else:
    roc_data = {}

# Compile all results
all_results = {
    **metrics,
    'roc': roc_data,
    'calibration': cal,
    'adversarial': adv_results,
    'kernel_attribution': kernel_attr,
    'alerts': alert_summary,
}

viz = ResultVisualizer(output_dir='../results')
viz.plot_all(
    results=all_results,
    train_metrics=trainer.metrics,
    X_sample=Xe[:500],
    y_sample=ye[:500],
    detector=detector,
    domain='multi',
)

---
## 14 — Summary Report

In [ ]:
print('=' * 70)
print('  EXPERIMENT SUMMARY')
print('=' * 70)
print(f'\n  Dataset       : ICS3D ({X_all.shape[0]:,} samples, {X_all.shape[1]} features)')
print(f'  Model         : Deep-Kernel Hierarchical GP')
print(f'  Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Inducing pts  : {model.gp.variational_strategy.inducing_points.shape[0]}')
print()
print('  DETECTION PERFORMANCE')
print('  ' + '-' * 30)
for k, v in metrics.items():
    print(f'    {k:12s}: {v:.4f}')
print(f'    {"ECE":12s}: {cal["ece"]:.4f}')
print()
print('  ADVERSARIAL ROBUSTNESS')
print('  ' + '-' * 30)
for eps, acc in adv_results.items():
    print(f'    eps={eps:.3f} : {acc:.4f}')
print()
print('  ALERT TIERS (sample 1000)')
print('  ' + '-' * 30)
for tier, cnt in alert_summary.items():
    print(f'    {tier:10s}: {cnt}')
print()
print('  TOP KERNEL COMPONENTS')
print('  ' + '-' * 30)
for name, frac in sorted(kernel_attr.items(), key=lambda x: -x[1])[:5]:
    print(f'    {name:30s}: {frac:.3f}')
print()
print('=' * 70)
print('  Results saved to: ../results/')
print('=' * 70)